# Week 7 demos

In [ ]:
import gdown
from urllib.request import urlretrieve

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cmocean as cmo

## Xarray

### Downloading, opening, and inspecting a netCDF file

In [ ]:
# Downloading .nc data file from google drive
#gdown.download(id="1jOXqaiSkXoGcB2G0ezHa50nVqeqGE23o", output="data/ofes_ts.nc")

In [ ]:
# load the .nc file as an xarray dataset
data_raw = xr.open_dataset('data/ofes_ts.nc')

In [ ]:
# inspect the data using display()
display(data_raw)

<!-- BEGIN QUESTION -->

### Poll #1

Based on summary of the dataset displayed above, which of he following is FALSE?

+ The `salinity` values are measured in PSU
+ There are 17920000 values (some may be `NaN`s) in `temp`
+ The span of `lat` is from 4.95°S to 4.95°N
+ The resolution (spacing) of `lev` is 5.0 m


<!-- END QUESTION -->

### Data cleaning and extraction

In [ ]:
# rename a dimension
data = data_raw.rename({'lev': 'depth'})
data

In [ ]:
# extract a single data variable

salinity = data['salinity']
salinity

In [ ]:
# extract a coordinates
depth = data.coords["depth"]
depth

In [ ]:
# extract the bare numpy arrays
depth_np = depth.values
depth_np

In [ ]:
salinity_np = salinity.values
salinity_np.shape

### Data manipulation

In [ ]:
# calculate the density of the seawater using an approximate linear formula
density = (
    1025.27
    + 0.766 * (data["salinity"] - 35.0)
    - 0.247 * (data["temp"] - 18.0)
    + 0.00441 * data.coords["depth"]
)
density

<!-- BEGIN QUESTION -->

### Your turn #1

Convert the temperature from celcius to ferenheit, and assign the result to a new data variable called `temp_F`

In [ ]:
# Your code here


<!-- END QUESTION -->

### Data selection and reduction

In [ ]:
# select salinity at sea level, and average over time
# Result is a 2-axes DataArray

sal_mean_surf = salinity.mean('time').isel(depth = 0)
display(sal_mean_surf)

<!-- BEGIN QUESTION -->

### Poll #2

Let's say I am interested in the longitude averaged temperature at a depth of (approximately) 10 m. Which of the following will produce the correct DataArray?

+ `data.isel(depth=10).mean("lon")`
+ `data["temp"].isel(depth=10).mean("lon")`
+ `data["temp"].sel(depth=10).mean("lon")`
+ `data["temp"].sel(depth=10, method="nearest").mean("lon")`

<!-- END QUESTION -->

## Reshaping multi-dimensional array redux

In [ ]:
file_url = 'https://raw.githubusercontent.com/OCEAN-215-2026/week_07_demos/refs/heads/main/data/temperature_grid.csv'
#urlretrieve(file_url, "data/temperature_grid.csv")

In [ ]:
temp_df = pd.read_csv("data/temperature_grid.csv")
display(temp_df)

In [ ]:
temp_df = temp_df.set_index(["time", "latitude", "longitude"])
display(temp_df)

In [ ]:
temp_xr = xr.Dataset.from_dataframe(temp_df)
display(temp_xr)

In [ ]:
temp_xr["temp_C"].values

In [ ]:
temp_xr.coords["time"].values

In [ ]:
temp_xr.coords["latitude"].values

In [ ]:
temp_xr.coords["longitude"].values

In [ ]:
temp_xr.mean(["time", "longitude"])

## Plotting 2D gridded data

In [ ]:
# Example: plotting the time-averaged sea-surface salinity as function of lat and lon

sal_mean_surf = salinity.mean('time').isel(depth = 0)
display(sal_mean_surf)

In [ ]:
# axis 0 is lat, axis 1 is lon (Good)
sal_array = sal_mean_surf.values

# extract lat and lon array
lat = sal_mean_surf.coords["lat"].values
lon = sal_mean_surf.coords["lon"].values

In [ ]:
# Now we can do the plotting
# starter code (no color bar)

fig = plt.figure()
ax = fig.add_subplot()

ax.pcolormesh(lon, lat, sal_array)
ax.set_aspect(1)

ax.set_title('Time-averaged sea-surface salinity') 
ax.set_xlabel('Longitude (°E)') 
ax.set_ylabel('Latitude (°N)')

plt.show(fig)

In [ ]:
# Including color bar


In [ ]:
# The same data, but as a filled contour plot


In [ ]:
# The same data, but as a line contour plot


In [ ]:
# Overlay salinity line contours on temperature false-color plot

temp_array = data["temp"].mean('time').isel(depth = 0).values

fig = plt.figure()
ax = fig.add_subplot()

im = ax.pcolormesh(lon, lat, temp_array, cmap=cmo.cm.thermal)

cb = plt.colorbar(im)
cb.set_label('Temperature (°C)')

cont = ax.contour(lon, lat, sal_array, cmap=cmo.cm.haline, linestyles="--")
clines = cont.clabel()

cb2 = plt.colorbar(cont)
cb2.set_label('Salinity (PSU)')

ax.set_title('Time-averaged sea-surface salinity') 
ax.set_xlabel('Longitude (°E)') 
ax.set_ylabel('Latitude (°N)')

ax.set_aspect(1)

plt.show(fig)

<!-- BEGIN QUESTION -->

### Your turn #3

Let's plot the time-average temperature at the equator (0°N) as a function of longitude and depth, using the usual convention that depth increases downwards

In [ ]:
# starter code

temp_eq = data["temp"].sel(lat=0, method="nearest").mean("time")


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

### Your turn #4

Consider the time-average temperature at the equator (0°N) as before. But this time let's we are interested only in the isotherms (constant temperature lines) from 14°C to 26°C at 2°C intervals. Modify your plot appropriately.

In [ ]:
# starter code

temp_eq = data["temp"].sel(lat=0, method="nearest").mean("time")


<!-- END QUESTION -->

